# CAFA5 Graph Smoke and Full Training

Objective: run two reproducible graph-training jobs from the existing CAFA5 graph cache.

Success criteria:
- Validate the graph Python environment before training.
- Create and train on a tiny smoke split so failures show up quickly.
- Run a standard full-split training job without changing the frozen graph split manifests.
- Save each run under a unique `graph_cache/training_runs/` folder and summarize the JSON metrics.

This notebook still reports the training script's built-in `micro_f1`. CAFA-style `wFmax` evaluation should be added as a separate prediction/evaluation step after the training output format is frozen.

In [ ]:
# Setup and configuration
from __future__ import annotations

import json
import os
import shlex
import subprocess
import sys
from datetime import datetime
from pathlib import Path

try:
    import pandas as pd
except ImportError:  # keep the notebook runnable in lean environments
    pd = None

from IPython.display import display

VALID_ASPECTS = {'BPO', 'CCO', 'MFO'}


def find_repo_root(start: Path) -> tuple[Path, str]:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate, 'cwd_search'
    return start, 'cwd_fallback'


def resolve_repo_root() -> tuple[Path, str]:
    configured = os.environ.get('CAFA5_REPO_ROOT')
    if configured:
        return Path(configured).expanduser().resolve(), 'env:CAFA5_REPO_ROOT'
    return find_repo_root(Path.cwd())


def parse_aspects(value: str) -> list[str]:
    parsed = [part.strip().upper() for part in value.replace(';', ',').split(',') if part.strip()]
    invalid = [aspect for aspect in parsed if aspect not in VALID_ASPECTS]
    if invalid:
        raise ValueError(f'Invalid aspects: {invalid}; expected subset of {sorted(VALID_ASPECTS)}')
    if not parsed:
        raise ValueError('At least one aspect is required.')
    return parsed


def env_bool(name: str, default: str = '0') -> bool:
    return os.environ.get(name, default).strip().lower() in {'1', 'true', 'yes', 'y'}


REPO_ROOT, REPO_ROOT_SOURCE = resolve_repo_root()
SERVER_USER = os.environ.get('USER', 'bensonli')
RUN_ROOT = Path(os.environ.get('CAFA5_RUN_ROOT', f'/global/scratch/users/{SERVER_USER}/cafa5_outputs'))
GRAPH_CACHE_DIR = RUN_ROOT / 'graph_cache'
GRAPH_SPLIT_DIR = GRAPH_CACHE_DIR / 'splits'
GRAPH_PYTHON = Path(os.environ.get('CAFA5_GRAPH_PYTHON', sys.executable))

NOTEBOOK_RUN_ID = os.environ.get('CAFA5_TRAIN_RUN_ID') or globals().get('NOTEBOOK_RUN_ID') or datetime.now().strftime('%Y%m%d_%H%M%S')
FRAMEWORK = os.environ.get('CAFA5_TRAIN_FRAMEWORK', 'pyg')
ASPECT = os.environ.get('CAFA5_TRAIN_ASPECT', 'MFO').upper()
FULL_ASPECTS = parse_aspects(os.environ.get('CAFA5_FULL_ASPECTS', ASPECT))
MIN_TERM_FREQUENCY = int(os.environ.get('CAFA5_MIN_TERM_FREQUENCY', '20'))
DEVICE = os.environ.get('CAFA5_TRAIN_DEVICE', 'auto')
SEED = int(os.environ.get('CAFA5_TRAIN_SEED', '2026'))

SMOKE_TRAIN_LIMIT = int(os.environ.get('CAFA5_SMOKE_TRAIN_LIMIT', '256'))
SMOKE_VAL_LIMIT = int(os.environ.get('CAFA5_SMOKE_VAL_LIMIT', '64'))
SMOKE_TEST_LIMIT = int(os.environ.get('CAFA5_SMOKE_TEST_LIMIT', '64'))
SMOKE_EPOCHS = int(os.environ.get('CAFA5_SMOKE_EPOCHS', '2'))
SMOKE_BATCH_SIZE = int(os.environ.get('CAFA5_SMOKE_BATCH_SIZE', '8'))
SMOKE_HIDDEN_DIM = int(os.environ.get('CAFA5_SMOKE_HIDDEN_DIM', '64'))
SMOKE_NUM_WORKERS = int(os.environ.get('CAFA5_SMOKE_NUM_WORKERS', '0'))

FULL_EPOCHS = int(os.environ.get('CAFA5_FULL_EPOCHS', '10'))
FULL_BATCH_SIZE = int(os.environ.get('CAFA5_FULL_BATCH_SIZE', '8'))
FULL_HIDDEN_DIM = int(os.environ.get('CAFA5_FULL_HIDDEN_DIM', '128'))
FULL_NUM_WORKERS = int(os.environ.get('CAFA5_FULL_NUM_WORKERS', '2'))

RUN_SMOKE_TRAINING = env_bool('CAFA5_RUN_SMOKE_TRAINING', '1')
RUN_FULL_TRAINING = env_bool('CAFA5_RUN_FULL_TRAINING', '1')
OVERWRITE_SMOKE_SPLITS = env_bool('CAFA5_OVERWRITE_SMOKE_SPLITS', '0')
USE_ESM2 = env_bool('CAFA5_USE_ESM2', '1')
USE_DSSP = env_bool('CAFA5_USE_DSSP', '1')
USE_SASA = env_bool('CAFA5_USE_SASA', '1')

SMOKE_SPLIT_DIR = GRAPH_CACHE_DIR / f'splits_smoke_{ASPECT.lower()}_t{SMOKE_TRAIN_LIMIT}_v{SMOKE_VAL_LIMIT}_s{SMOKE_TEST_LIMIT}'
SMOKE_CHECKPOINT_DIR = GRAPH_CACHE_DIR / 'training_runs' / 'notebook_smoke' / NOTEBOOK_RUN_ID / f'{FRAMEWORK}_{ASPECT.lower()}'
FULL_CHECKPOINT_ROOT = GRAPH_CACHE_DIR / 'training_runs' / 'notebook_full' / NOTEBOOK_RUN_ID

config = {
    'repo_root': str(REPO_ROOT),
    'repo_root_source': REPO_ROOT_SOURCE,
    'run_root': str(RUN_ROOT),
    'graph_python': str(GRAPH_PYTHON),
    'graph_cache_dir': str(GRAPH_CACHE_DIR),
    'graph_split_dir': str(GRAPH_SPLIT_DIR),
    'notebook_run_id': NOTEBOOK_RUN_ID,
    'framework': FRAMEWORK,
    'aspect_for_smoke': ASPECT,
    'full_aspects': FULL_ASPECTS,
    'min_term_frequency': MIN_TERM_FREQUENCY,
    'device': DEVICE,
    'seed': SEED,
    'smoke': {
        'split_dir': str(SMOKE_SPLIT_DIR),
        'checkpoint_dir': str(SMOKE_CHECKPOINT_DIR),
        'limits': {'train': SMOKE_TRAIN_LIMIT, 'val': SMOKE_VAL_LIMIT, 'test': SMOKE_TEST_LIMIT},
        'epochs': SMOKE_EPOCHS,
        'batch_size': SMOKE_BATCH_SIZE,
        'hidden_dim': SMOKE_HIDDEN_DIM,
        'num_workers': SMOKE_NUM_WORKERS,
        'run_training': RUN_SMOKE_TRAINING,
    },
    'full': {
        'split_dir': str(GRAPH_SPLIT_DIR),
        'checkpoint_root': str(FULL_CHECKPOINT_ROOT),
        'epochs': FULL_EPOCHS,
        'batch_size': FULL_BATCH_SIZE,
        'hidden_dim': FULL_HIDDEN_DIM,
        'num_workers': FULL_NUM_WORKERS,
        'run_training': RUN_FULL_TRAINING,
    },
    'modalities': {'esm2': USE_ESM2, 'dssp': USE_DSSP, 'sasa': USE_SASA},
}
print(json.dumps(config, indent=2))

required_repo_files = [REPO_ROOT / 'train_minimal_graph_model.py', REPO_ROOT / 'cafa_graph_dataloaders.py']
missing_repo_files = [str(path) for path in required_repo_files if not path.exists()]
if missing_repo_files:
    raise FileNotFoundError(f'REPO_ROOT is wrong or incomplete: {missing_repo_files}')
if ASPECT not in VALID_ASPECTS:
    raise ValueError(f'Invalid smoke aspect: {ASPECT}')

## Plan

1. Check that `GRAPH_PYTHON` can import the training backend.
2. Build a small smoke split from the frozen graph split files and run a short training job.
3. Run the standard full training job on `graph_cache/splits` with `min_term_frequency=20`.
4. Read each `summary.json` and compare the final train/val/test metrics.

For a report-ready run across all three namespaces, set `CAFA5_FULL_ASPECTS=BPO,CCO,MFO` before launching the notebook or edit `FULL_ASPECTS` in the setup cell.

The full run executes by default. Set `CAFA5_RUN_FULL_TRAINING=0` before launching the notebook if you only want to preview commands or if you are on a login node.

In [ ]:
# Shared helpers for command execution, tables, and training summaries.
def format_cmd(cmd: list[object]) -> str:
    return ' '.join(shlex.quote(str(part)) for part in cmd)


def run_live(cmd: list[object], cwd: Path | None = None) -> None:
    cmd_str = format_cmd(cmd)
    print(f'[run] {cmd_str}', flush=True)
    process = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=str(cwd or REPO_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, [str(part) for part in cmd])


def show_records(records: list[dict]) -> None:
    if pd is not None:
        display(pd.DataFrame(records))
    else:
        print(json.dumps(records, indent=2))


def read_entry_ids(path: Path) -> list[str]:
    return [line.strip() for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]


def write_lines_if_needed(path: Path, lines: list[str], overwrite: bool = False) -> None:
    if path.exists() and not overwrite:
        print(f'[skip] already exists: {path}')
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{line}\n' for line in lines), encoding='utf-8')
    print(f'[write] {path} ({len(lines)} lines)')


def write_json_if_needed(path: Path, payload: dict, overwrite: bool = False) -> None:
    if path.exists() and not overwrite:
        print(f'[skip] already exists: {path}')
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    print(f'[write] {path}')


def modality_flags() -> list[str]:
    flags = []
    if not USE_ESM2:
        flags.append('--disable-esm2')
    if not USE_DSSP:
        flags.append('--disable-dssp')
    if not USE_SASA:
        flags.append('--disable-sasa')
    return flags


def build_train_cmd(
    *,
    aspect: str,
    split_dir: Path,
    checkpoint_dir: Path,
    epochs: int,
    batch_size: int,
    num_workers: int,
    hidden_dim: int,
) -> list[object]:
    return [
        GRAPH_PYTHON,
        REPO_ROOT / 'train_minimal_graph_model.py',
        '--root', GRAPH_CACHE_DIR,
        '--split-dir', split_dir,
        '--framework', FRAMEWORK,
        '--aspect', aspect,
        '--epochs', epochs,
        '--batch-size', batch_size,
        '--num-workers', num_workers,
        '--hidden-dim', hidden_dim,
        '--device', DEVICE,
        '--seed', SEED,
        '--min-term-frequency', MIN_TERM_FREQUENCY,
        '--checkpoint-dir', checkpoint_dir,
        *modality_flags(),
    ]


def summarize_training_summary(summary_path: Path, run_name: str, aspect: str) -> list[dict]:
    if not summary_path.exists():
        print(f'[missing] {summary_path}')
        return []
    training_summary = json.loads(summary_path.read_text(encoding='utf-8'))
    rows = []
    for record in training_summary.get('history', []):
        row = {
            'run': run_name,
            'aspect': aspect,
            'epoch': record.get('epoch'),
            'epoch_seconds': record.get('epoch_seconds'),
        }
        for split_name in ('train', 'val', 'test'):
            metrics = record.get(split_name, {})
            row[f'{split_name}_loss'] = metrics.get('loss')
            row[f'{split_name}_micro_f1'] = metrics.get('micro_f1')
            row[f'{split_name}_graphs'] = metrics.get('graphs')
        rows.append(row)
    key_summary = {
        'run': run_name,
        'aspect': aspect,
        'summary_path': str(summary_path),
        'best_val_loss': training_summary.get('best_val_loss'),
        'best_checkpoint_path': training_summary.get('best_checkpoint_path'),
    }
    print(json.dumps(key_summary, indent=2))
    return rows


def latest_rows(rows: list[dict]) -> list[dict]:
    by_run = {}
    for row in rows:
        by_run[(row['run'], row['aspect'])] = row
    return list(by_run.values())

In [ ]:
# Validate the graph training environment and required artifacts before training.
backend_import = 'torch_geometric' if FRAMEWORK == 'pyg' else 'dgl'
env_check = [
    GRAPH_PYTHON,
    '-c',
    (
        'import importlib, sys; import torch; '
        f'{backend_import}=importlib.import_module("{backend_import}"); '
        'print("python", sys.executable); '
        'print("torch", torch.__version__); '
        f'print("{backend_import}", getattr({backend_import}, "__version__", "unknown")); '
        'print("cuda_available", torch.cuda.is_available())'
    ),
]
run_live(env_check)

required_artifacts = [GRAPH_CACHE_DIR, GRAPH_SPLIT_DIR, GRAPH_SPLIT_DIR / 'summary.json']
missing_artifacts = [str(path) for path in required_artifacts if not path.exists()]
if missing_artifacts:
    raise FileNotFoundError(f'Missing graph artifacts: {missing_artifacts}')

for aspect in sorted({ASPECT, *FULL_ASPECTS}):
    aspect_dir = GRAPH_SPLIT_DIR / aspect.lower()
    for name in ('train.txt', 'val.txt', 'test.txt'):
        path = aspect_dir / name
        if not path.exists():
            raise FileNotFoundError(f'Missing split file for {aspect}: {path}')
print('[ok] graph environment and split files are ready')

## Smoke Training

This step creates a tiny split under `graph_cache/splits_smoke_*` and trains for a couple of epochs. It is meant to catch environment, dataloader, and model-shape problems before the full run.

In [ ]:
# Build the smoke split from the frozen graph split manifests.
source_aspect_dir = GRAPH_SPLIT_DIR / ASPECT.lower()
smoke_aspect_dir = SMOKE_SPLIT_DIR / ASPECT.lower()
smoke_limits = {'train': SMOKE_TRAIN_LIMIT, 'val': SMOKE_VAL_LIMIT, 'test': SMOKE_TEST_LIMIT}

smoke_summary = {
    'source_split_dir': str(source_aspect_dir),
    'root': str(SMOKE_SPLIT_DIR),
    'aspect': ASPECT,
    'min_term_frequency': MIN_TERM_FREQUENCY,
    'seed': SEED,
    'limits': smoke_limits,
    'counts': {},
    'source_counts': {},
}

for split_name, limit in smoke_limits.items():
    src = source_aspect_dir / f'{split_name}.txt'
    ids = read_entry_ids(src)
    subset = ids[:min(limit, len(ids))]
    smoke_summary['source_counts'][split_name] = len(ids)
    smoke_summary['counts'][split_name] = len(subset)
    write_lines_if_needed(smoke_aspect_dir / f'{split_name}.txt', subset, overwrite=OVERWRITE_SMOKE_SPLITS)

source_summary_path = source_aspect_dir / 'summary.json'
if source_summary_path.exists():
    write_lines_if_needed(
        smoke_aspect_dir / 'summary.json',
        source_summary_path.read_text(encoding='utf-8').splitlines(),
        overwrite=OVERWRITE_SMOKE_SPLITS,
    )
write_json_if_needed(SMOKE_SPLIT_DIR / 'summary.json', smoke_summary, overwrite=OVERWRITE_SMOKE_SPLITS)

show_records([
    {
        'split': split_name,
        'selected': smoke_summary['counts'][split_name],
        'source_total': smoke_summary['source_counts'][split_name],
    }
    for split_name in ('train', 'val', 'test')
])

In [ ]:
# Execute the smoke training run.
smoke_cmd = build_train_cmd(
    aspect=ASPECT,
    split_dir=SMOKE_SPLIT_DIR,
    checkpoint_dir=SMOKE_CHECKPOINT_DIR,
    epochs=SMOKE_EPOCHS,
    batch_size=SMOKE_BATCH_SIZE,
    num_workers=SMOKE_NUM_WORKERS,
    hidden_dim=SMOKE_HIDDEN_DIM,
)

if RUN_SMOKE_TRAINING:
    run_live(smoke_cmd)
else:
    print('[skip] RUN_SMOKE_TRAINING is False')
    print(format_cmd(smoke_cmd))

In [ ]:
# Summarize the smoke run.
smoke_rows = summarize_training_summary(
    SMOKE_CHECKPOINT_DIR / 'summary.json',
    run_name='smoke',
    aspect=ASPECT,
)
show_records(smoke_rows)

## Standard Full Training

This run uses the full existing split directory: `graph_cache/splits`. It does not rewrite split files. The default is one aspect (`MFO`) with `min_term_frequency=20`, which matches the first stable full-data policy from the experiment plan.

For all three official namespaces, set `FULL_ASPECTS = ['BPO', 'CCO', 'MFO']` or run with `CAFA5_FULL_ASPECTS=BPO,CCO,MFO`.

In [ ]:
# Build and optionally execute the standard full-split training commands.
full_runs = []
for aspect in FULL_ASPECTS:
    checkpoint_dir = FULL_CHECKPOINT_ROOT / f'{FRAMEWORK}_{aspect.lower()}_mtf{MIN_TERM_FREQUENCY}'
    cmd = build_train_cmd(
        aspect=aspect,
        split_dir=GRAPH_SPLIT_DIR,
        checkpoint_dir=checkpoint_dir,
        epochs=FULL_EPOCHS,
        batch_size=FULL_BATCH_SIZE,
        num_workers=FULL_NUM_WORKERS,
        hidden_dim=FULL_HIDDEN_DIM,
    )
    full_runs.append({'aspect': aspect, 'checkpoint_dir': checkpoint_dir, 'cmd': cmd})

for run in full_runs:
    print('\n# Full training command:', run['aspect'])
    print(format_cmd(run['cmd']))

if RUN_FULL_TRAINING:
    for run in full_runs:
        run_live(run['cmd'])
else:
    print('\n[skip] RUN_FULL_TRAINING is False')
    print('Set CAFA5_RUN_FULL_TRAINING=1 or edit the setup cell to launch the full run.')

In [ ]:
# Summarize the full training runs that have completed.
full_rows = []
for run in full_runs:
    full_rows.extend(
        summarize_training_summary(
            run['checkpoint_dir'] / 'summary.json',
            run_name='full',
            aspect=run['aspect'],
        )
    )
show_records(full_rows)

## Final Comparison

Use the final epoch rows as a quick sanity check. For formal analysis, export prediction scores and run CAFA-style evaluation (`wFmax`) instead of relying only on fixed-threshold `micro_f1`.

In [ ]:
# Compare final-epoch metrics across completed runs.
combined_rows = latest_rows(smoke_rows + full_rows)
show_records(combined_rows)

## Notes

Record observations here after the run:

- Did the smoke run finish without dataloader/model errors?
- Did the full run use the intended split, aspect, and `min_term_frequency`?
- Did GPU/CUDA show up in the environment check when expected?
- Which checkpoint path should be used for prediction export and CAFA-style evaluation next?